Improvement: add rolling averages??

In [0]:
ROOT = "/Workspace/Users/melaniepieterse1@gmail.com/Loan_Offer_Model"
CATALOG_PATH = f"{ROOT}/conf/base/catalog.yml"
import sys
sys.path.insert(0, ROOT)
from src.catalog_loader import Catalog
catalog = Catalog(spark, CATALOG_PATH)

customers_df = catalog.load("prep_customer")
prep_loans_df = catalog.load("loans")
prep_transactions_df = catalog.load("prep_transactions")

print(f"customers_df: {customers_df.count()} rows")
print(f"loans_df: {prep_loans_df.count()} rows")
print(f"transactions_df: {prep_transactions_df.count()} rows")
print(f"customers_df: {customers_df.count()} rows, {customers_df.select('ID').distinct().count()} unique IDs")
print(f"loans_df: {prep_loans_df.count()} rows, {prep_loans_df.select('ID').distinct().count()} unique IDs")
print(f"transactions_df: {prep_transactions_df.count()} rows, {prep_transactions_df.select('ID').distinct().count()} unique IDs")

In [0]:
from pyspark.sql.functions import when, col, count, datediff, lag, avg, mode, lit,  min, max, date_format, sum,weekofyear, max_by, std
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import skew
import matplotlib.pyplot as plt
sns.set(style="whitegrid", palette="muted", font_scale=1.1)

In [0]:
# Determine the range of data  - data is availble over 4 months
date_range = prep_transactions_df.agg(min("DATE").alias("min_date"), max("DATE").alias("max_date"),)
display(date_range)

In [0]:
analysis_level = 'MONTH'  # 'WEEK' #'DAY

In [0]:
loans_df = prep_loans_df

w_outcome = Window.partitionBy("ID")
w_date = Window.partitionBy("ID").orderBy("DATE")

loans_df = loans_df.withColumn( "DECLINED_OUTCOME", count(when(col("OUTCOME") == "Declined", 1)).over(w_outcome)
).withColumn("SUCCESS_OUTCOME", count(when(col("OUTCOME") == "TakeUp", 1)).over(w_outcome)).withColumn("INTER_OFFER_TIME_DAYS", datediff(col("DATE"), lag("DATE").over(w_date)))

loans_df = loans_df.groupBy('ID').agg(count('*').alias('TOTAL_OFFERS'), avg('INTER_OFFER_TIME_DAYS').alias('AVG_INTER_OFFER_TIME_DAYS'), mode('DECLINED_OUTCOME').alias('DECLINED_OUTCOMES'), mode('SUCCESS_OUTCOME').alias('SUCCESS_OUTCOME'),avg('AMOUNT').alias('AVG_AMOUNT_OFFERED'), max('DATE').alias('LAST_OFFER_DATE'))

loans_df = loans_df.withColumn('DAYS_SINCE_LAST_OFFER', datediff(lit('2022-08-30'), col('LAST_OFFER_DATE'))).drop('LAST_OFFER_DATE')

loans_df = loans_df.withColumn('LABEL', when(col('SUCCESS_OUTCOME') > 0, 1).otherwise(0))

display(loans_df)


In [0]:
transactions_df = prep_transactions_df.join(customers_df, on='ID', how ='full')

transactions_df = transactions_df.withColumn('MONTH', date_format("DATE", "MM")).withColumn('WEEK', weekofyear("DATE")).withColumn('DAY', date_format("DATE", "dd"))


# Count the number of postive and negative analysis_level end balances. -all balances result in positive.

analysis_level_transactions = transactions_df.groupBy('ID', analysis_level, 'GENDER', 'AGE', 'INCOME').agg(max_by('BALANCE', '__index_level_0__').alias('END_BALANCE'),
    sum(when(col('AMOUNT') < 0, 1).otherwise(0)).alias('TRANS_COUNT'),
    sum(when(col('AMOUNT') > 0, 1).otherwise(0)).alias('INCOME_COUNT'),
    avg(when(col('AMOUNT') < 0, col('AMOUNT')).otherwise(lit(None))).alias('AVG_TRANS_AMOUNT'),
    sum(when(col('AMOUNT') > 0, col('AMOUNT')).otherwise(lit(None))).alias('TOTAL_INCOME'),
    sum(when(col('AMOUNT') < 0, col('AMOUNT')).otherwise(lit(None))).alias('TOTAL_EXPENSES'),
    avg(when(col('AMOUNT') == col('INCOME'), col('DAY').cast('int')).otherwise(lit(None))).alias('INCOME_DAY'),
    sum(when((col('AMOUNT') < 0) & (col('BALANCE') < 0), 1).otherwise(0)).alias('OVERDRAFT_SPENT'),
    sum(when((col('AMOUNT') < 0) & (col('BALANCE') < 0), col('AMOUNT')).otherwise(lit(None))).alias('OVERDRAFT_AMOUNT'))

analysis_level_transactions = analysis_level_transactions.withColumn('ADD_INCOME', when(col('TOTAL_INCOME') == 0, lit(None)).otherwise(when(col('TOTAL_INCOME') > col('INCOME'), col('TOTAL_INCOME') - col('INCOME')).otherwise(0))).withColumn('LIVING_MEANS', when(col('TOTAL_INCOME') == 0, lit(None)).otherwise(col('TOTAL_INCOME') - col('TOTAL_EXPENSES'))).withColumn('INCOME_SPENT_%', when(col('TOTAL_INCOME') == 0, lit(None)).otherwise((-col('TOTAL_EXPENSES') / col('TOTAL_INCOME')) * 100))

display(analysis_level_transactions)


transaction_summary = analysis_level_transactions.groupBy('ID','GENDER','AGE','INCOME').agg(sum(when(col('END_BALANCE') > 0, 1).otherwise(0)).alias('POSITIVE_END_BALANCE'),sum(when(col('END_BALANCE') < 0, 1).otherwise(0)).alias('NEGATIVE_END_BALANCE'), avg('END_BALANCE').alias('AVG_MONTHLY_END_BALANCE'), std('END_BALANCE').alias('STD_MONTHLY_END_BALANCE'),avg('TRANS_COUNT').alias('AVG_TRANS_COUNT'),std('TRANS_COUNT').alias('STD_TRANS_COUNT'),avg('INCOME_COUNT').alias('AVG_INCOME_COUNT'),std('INCOME_COUNT').alias('STD_INCOME_COUNT'),avg('AVG_TRANS_AMOUNT').alias('AVG_TRANS_AMOUNT'),std('AVG_TRANS_AMOUNT').alias('STD_TRANS_AMOUNT'),avg('TOTAL_INCOME').alias('AVG_TOTAL_INCOME'),std('TOTAL_INCOME').alias('STD_TOTAL_INCOME'),avg('TOTAL_EXPENSES').alias('AVG_TOTAL_EXPENSES'),std('TOTAL_EXPENSES').alias('STD_TOTAL_EXPENSES'),mode('INCOME_DAY').alias('INCOME_DAY'),avg('OVERDRAFT_AMOUNT').alias('AVG_OVERDRAFT_SPENT'),sum('OVERDRAFT_SPENT').alias('OVERDRAFT_SPENT'),avg('ADD_INCOME').alias('AVG_ADD_INCOME'),sum('ADD_INCOME').alias('ADD_INCOME'), avg('LIVING_MEANS').alias('LIVING_MEANS'),avg('INCOME_SPENT_%').alias('INCOME_SPENT_%')).fillna(0)

display(transaction_summary)

In [0]:
final_df = transaction_summary.join(loans_df, on ="ID", how = 'outer').fillna(0)
display(final_df)

final_df.write.format("delta").mode("overwrite").save(f"{ROOT}/data/03_primary/consolidated_data")

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

# Considerations
- There is a class imbalance. More people have not accepted a loan offering than people who has. The evaluation metric used will take the class imbalance into consederation.
- There are more or less the same amount of females and males - 50.5% males and  49.5% females. However, more females have accepted a loan offer (65%) than males. However, looking at loans that were offered, 56% of loans were offered to men and  44% to women. Women seems to be more likely to pick-up a loan offer based on the above opservation.

In [0]:
if not isinstance(final_df, pd.DataFrame):
    final_df = pd.DataFrame([row.asDict() for row in final_df.collect()])
final_df['GENDER'] = final_df['GENDER'].map({'M': 0, 'F': 1})

In [0]:
%skip
# Are men more likely to have loans approved? This tells you if the difference is statistically significant.
from scipy.stats import ttest_ind
# GENDER_0 = M, GENDER_1 = F
ttest_ind(final_df[final_df.GENDER_1==1].AVG_AMOUNT_OFFERED,final_df[final_df.GENDER_0==1].AVG_AMOUNT_OFFERED,nan_policy='omit')

# The p‑value (0.0073)
# This is the most important part.
# p < 0.05 → statistically significant difference
# p = 0.0073 → highly significant
# This means the difference in average loan amounts offered between men and women is unlikely to be due to chance. There is a statistically significant gender difference in AVG_AMOUNT_OFFERED.
# Females have a LOWER average loan amount offered than males because the t‑statistic (-2.6836) is negative indicating which group has the higher mean

In [0]:
#%pip install missingno
# Missing values summary
missing = final_df.isnull().sum().sort_values(ascending=False)
missing_pct = (final_df.isnull().mean()*100).sort_values(ascending=False)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
import missingno as msno
#msno.matrix(final_df)
#msno.bar(final_df)
missing_df


In [0]:
# Remove highly correlated features after repetion
# Keep ID and LABEL for reference, but exclude from all analytics/skewness
cols_to_exclude = ['ID', 'LABEL']
df = final_df.copy()                     # retains ID and LABEL
df_features = df.drop(columns=cols_to_exclude)  # features only, for correlation
plt.figure(figsize=(14,10))
corr = df_features.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap Before Removing Redundant Features")
plt.show()

# Correlation matrix
corr_matrix = df_features.corr().abs()

# Select upper triangle (to avoid duplicates)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find pairs with correlation > 0.8
high_corr = [(column, row, upper.loc[row, column]) 
             for column in upper.columns 
             for row in upper.index 
             if upper.loc[row, column] > 0.80]

redundant_features = [pair[0] for pair in high_corr]
df_reduced = df.drop(columns=redundant_features)   # still keeps ID and LABEL

plt.figure(figsize=(14,10))
corr = df_reduced.drop(columns=cols_to_exclude).corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap After Removing Redundant Features")
plt.show()

high_corr

In [0]:
numeric_cols = df_reduced.select_dtypes(include=['float64','int64']).columns
for col in numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(final_df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

| Skewness | Direction | Action |
|---|---|---|
| > 1 | positive | log1p transform (strong right skew) |
| 0.5–1 | positive | sqrt transform (moderate right skew) |
| 0–0.5 | positive | no transform (light skew) |
| < -1 | negative | reflect + log transform (strong left skew) |
| -1–0 | negative | square transform (moderate left skew) |


Do not transform:
- binary features (0/1)
- categorical encoded features
- percentages already between 0–1
- features with light skew (|skew| < 0.5)
- features where interpretability matters (e.g., AGE)


In [0]:
# Create a dictionary to store the skewness of each numeric column
skewness_results = []
cols_to_exclude = ['ID', 'LABEL','AGE','INCOME_SPENT_%','GENDER']
skew_cols = [c for c in numeric_cols if c not in cols_to_exclude]
for c in skew_cols:
    s = skew(df_reduced[c].dropna())
    direction = 'positive' if s > 0 else 'negative' if s < 0 else 'none'
    skewness_results.append({'feature': c, 'skewness': s})

skew_dict = pd.DataFrame(skewness_results).sort_values('skewness', ascending=False)
display(skew_dict)
skew_dict = dict(zip(skew_dict['feature'], skew_dict['skewness']))

def transform_by_skew(df, skew_dict):
    """
    Automatically transforms columns based on skewness direction and magnitude.
    
    Parameters:
    df (pd.DataFrame): Your dataframe
    skew_dict (dict): {column_name: skewness_value}
    
    Returns:
    pd.DataFrame: transformed dataframe
    """
    
    df_transformed = df.copy()
    
    for col, skew in skew_dict.items():
        # Skip non-numeric columns
        if not np.issubdtype(df[col].dtype, np.number):
            continue
        
        # Strong positive skew → log transform
        if skew > 1:
            df_transformed[col] = np.log1p(df[col].clip(lower=0))
            print(f"{col}: strong positive skew → log1p transform applied.")
        
        # Moderate positive skew → sqrt transform
        elif 0.5 < skew <= 1:
            df_transformed[col] = np.sqrt(df[col].clip(lower=0))
            print(f"{col}: moderate positive skew → sqrt transform applied.")
        
        # Light positive skew → no transform
        elif 0 < skew <= 0.5:
            print(f"{col}: light positive skew → no transform applied.")
        
        # Strong negative skew → reflect + log transform
        elif skew < -1:
            max_val = df[col].max()
            df_transformed[col] = np.log1p(max_val - df[col])
            print(f"{col}: strong negative skew → reflect + log transform applied.")
        
        # Moderate negative skew → square transform
        elif -1 <= skew < 0:
            df_transformed[col] = df[col] ** 2
            print(f"{col}: moderate negative skew → square transform applied.")
    
    return df_transformed

final_df_transformed = transform_by_skew(df_reduced, skew_dict)
display(final_df_transformed)

In [0]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.0)
selector.fit(final_df_transformed[numeric_cols])
zero_var_cols = [col for col, var in zip(numeric_cols, selector.variances_) if var == 0]
zero_var_cols

In [0]:
# Remove highly correlated features after repetion
# Keep ID and LABEL for reference, but exclude from all analytics/skewness
cols_to_exclude = ['ID', 'LABEL']
df = final_df_transformed.copy()                     # retains ID and LABEL
df_features = df.drop(columns=cols_to_exclude)  # features only, for correlation
plt.figure(figsize=(14,10))
corr = df_features.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap Before Removing Redundant Features")
plt.show()

# Correlation matrix
corr_matrix = df_features.corr().abs()

# Select upper triangle (to avoid duplicates)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find pairs with correlation > 0.85
high_corr = [(column, row, upper.loc[row, column]) 
             for column in upper.columns 
             for row in upper.index 
             if upper.loc[row, column] > 0.85]

redundant_features = [pair[0] for pair in high_corr]
df_reduced = df.drop(columns=redundant_features)   # still keeps ID and LABEL

plt.figure(figsize=(14,10))
corr = df_reduced.drop(columns=cols_to_exclude).corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap After Removing Redundant Features")
plt.show()

high_corr


In [0]:
feature_df = df_reduced
display(feature_df)

feature_df.write.format("delta").mode("overwrite").save(f"{ROOT}/data/04_primary/features")